# Module 6.5: from notebook to module

**Goal:** turn analysis code that lives in notebook cells into reusable, testable `.py` files, the shape your project needs.

**You will learn:** why and when to leave the notebook · scripts and `if __name__ == "__main__"` · importing your own code ·
packages and `src/` layout · refactoring cells into functions · paths · command-line arguments · tests with `pytest`

**Maths in this module:** none. The skill here is *structure*.

> 🖥️ This module works best on your own computer (see Module 0, option B). The lesson itself creates files only in a
> **temporary folder**, so you can run it freely without cluttering your project.

## 1. Why leave the notebook?

Notebooks are brilliant for exploring and for telling a story. But they have weak spots:

| notebook weakness | what goes wrong | fix |
|---|---|---|
| **hidden state**: cells can run out of order | results you can't reproduce | put logic in functions; run **Restart & Run All** |
| **copy-paste** between notebooks | a bug fixed in one place, alive in another | one function, imported everywhere |
| **hard to test** | you only notice mistakes by eye | `pytest` checks for functions |
| **hard to run unattended** | can't schedule or call from a terminal | a script / command |
| **messy version-control diffs** | JSON noise | `.py` files diff cleanly |

The pattern that works: **explore in notebooks → move working code into `.py` modules → import it back into notebooks.**

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="module65_"))
print("Playground folder:", workdir)

def show_tree(root, prefix=""):
    """Print a folder as an indented tree."""
    entries = sorted(p for p in Path(root).iterdir() if p.name != "__pycache__" and not p.name.startswith(".")
                     and not p.name.endswith(".egg-info"))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            show_tree(p, prefix + ("    " if last else "│   "))

def run(*args, cwd=workdir):
    """Run a command and print its output (like typing it in a terminal)."""
    result = subprocess.run(args, capture_output=True, text=True, cwd=cwd)
    print(result.stdout + result.stderr, end="")

## 2. A script and `if __name__ == "__main__"`

A **module** is just a `.py` file. You can *run* it, or *import* it from somewhere else. The special variable `__name__`
tells the file which of the two is happening, so one file can do both:

In [ ]:
(workdir / "temp_tools.py").write_text('''
def celsius_to_f(c):
    """Convert Celsius to Fahrenheit."""
    return 9 / 5 * c + 32


if __name__ == "__main__":
    # This block runs only when the file is executed directly, not when it is imported.
    print("running as a script:", celsius_to_f(100))
'''.lstrip())

run(sys.executable, "temp_tools.py")          # same as typing:  python temp_tools.py

In [ ]:
sys.path.insert(0, str(workdir))              # let Python look for modules in our playground

import temp_tools                             # importing does NOT trigger the __main__ block
print(temp_tools.celsius_to_f(0), "| module name:", temp_tools.__name__)

> ⚠️ **Name clashes:** never name your file after a library (`pandas.py`, `random.py`, `test.py`). Python would import
> *yours* instead of the real one and confusing errors follow.
>
> 🔄 **Editing a module you already imported?** Python caches imports. In a notebook run `%load_ext autoreload` and
> `%autoreload 2` once at the top, and edits to your `.py` files are picked up automatically.

## 3. From a script to a package

As the code grows, split it by job. A **package** is a folder of modules with an `__init__.py` file. The widely used
**`src/` layout** keeps importable code apart from everything else:

```
my-project/
├── pyproject.toml        <- describes the package (name, dependencies)
├── run_analysis.py       <- thin entry-point script
├── src/
│   └── myproject/
│       ├── __init__.py
│       ├── load.py       <- read data
│       ├── clean.py      <- cleaning functions
│       ├── analysis.py   <- statistics / models
│       └── plots.py      <- chart functions
├── tests/
└── notebooks/
```

Let's build a tiny one right now:

In [ ]:
pkg = workdir / "src" / "demo_pkg"
pkg.mkdir(parents=True)

(pkg / "__init__.py").write_text('"""A demo package."""\n')

(pkg / "stats.py").write_text('''
"""Basic statistics helpers."""


def mean(values):
    """Return the arithmetic mean of a non-empty list of numbers."""
    if not values:
        raise ValueError("mean() needs at least one value")
    return sum(values) / len(values)
'''.lstrip())

(pkg / "report.py").write_text('''
"""Uses another module of the same package."""
from demo_pkg.stats import mean


def summary(values):
    return f"n={len(values)}, mean={mean(values):.1f}"
'''.lstrip())

show_tree(workdir)

In [ ]:
sys.path.insert(0, str(workdir / "src"))

from demo_pkg.stats import mean
from demo_pkg import report

print(mean([1, 2, 3, 4]))
print(report.summary([72, 85, 90]))

Notice the **absolute import** `from demo_pkg.stats import mean` inside `report.py`: it works from anywhere.
Prefer these over tricks like editing `sys.path` in your own code.

### Installing your package: `pip install -e .`
Editing `sys.path` by hand (as we did above just for this demo) isn't how real projects do it. Instead, a small
`pyproject.toml` describes your package and you install it once in **editable mode**:

```bash
pip install -e .
```

Now `import myproject` works from any notebook or script in that environment, and edits to the source take effect immediately.
The project template ([`projects/template/`](../../projects/template/)) is already set up this way.

## 4. Refactoring cells into functions

Here's a typical notebook cell: it works, but it depends on global variables, mixes steps, and prints instead of returning.

In [ ]:
import pandas as pd

# ---- BEFORE: typical notebook code ----
data = pd.DataFrame({"city": [" bangkok", "Bangkok ", "Phuket", None], "sales": ["10", "20", "x", "5"]})

data["city"] = data["city"].str.strip().str.title()
data["sales"] = pd.to_numeric(data["sales"], errors="coerce")
data = data.dropna()
print(data)

**Refactoring rules of thumb**

1. **One job per function**, with a name that is a verb phrase (`clean_sales`, `load_rides`).
2. **Inputs are arguments, outputs are `return`s.** No reaching for global variables.
3. **Don't modify the input**: work on a `.copy()` and return the new frame.
4. **Return, don't print.** The caller decides what to show.
5. Add a **docstring** saying what goes in and what comes out.
6. Add **type hints** (`df: pd.DataFrame`) as documentation. They're optional but helpful.

In [ ]:
# ---- AFTER: reusable, testable ----
def clean_sales(df: pd.DataFrame) -> pd.DataFrame:
    """Tidy city names, convert sales to numbers and drop rows that are still incomplete."""
    out = df.copy()
    out["city"] = out["city"].str.strip().str.title()
    out["sales"] = pd.to_numeric(out["sales"], errors="coerce")
    return out.dropna().reset_index(drop=True)

raw = pd.DataFrame({"city": [" bangkok", "Bangkok ", "Phuket", None], "sales": ["10", "20", "x", "5"]})
clean_sales(raw)

## 5. Paths that work everywhere

Hard-coded paths like `"C:/Users/me/Desktop/data.csv"` break the moment someone else (or you, on another computer) runs your code.

* Build paths with **`pathlib.Path`** (works on Windows, macOS and Linux).
* Locate files **relative to your project**, not to wherever you happened to start Python.

In [ ]:
# In a .py file, __file__ is the path of that file. Project root = a known number of folders up:
#     ROOT = Path(__file__).resolve().parents[2]          # e.g. src/myproject/load.py -> project root
#     RAW = ROOT / "data" / "raw" / "orders.csv"
p = Path("data") / "raw" / "orders.csv"
print(p, "| name:", p.name, "| suffix:", p.suffix, "| parent:", p.parent)

## 6. Command-line arguments with `argparse`

A script that takes options can be reused without editing code: `python run_analysis.py --input other.csv`.

In [ ]:
(workdir / "greet.py").write_text('''
import argparse


def main(argv=None):
    parser = argparse.ArgumentParser(description="Say hello a few times.")
    parser.add_argument("--name", default="world")
    parser.add_argument("--times", type=int, default=1)
    args = parser.parse_args(argv)
    for _ in range(args.times):
        print(f"Hello, {args.name}!")


if __name__ == "__main__":
    main()
'''.lstrip())

run(sys.executable, "greet.py", "--name", "Ada", "--times", "2")
run(sys.executable, "greet.py", "--help")

Design tip: put the real work in a function `main(argv=None)` so tests can call it directly (the exercise does exactly this).

## 7. Testing with `pytest`

A **test** is a function whose name starts with `test_` and that uses `assert`. `pytest` finds and runs them all.
Tests on small, hand-made inputs catch mistakes that you'd never spot by eye in a 10,000-row table.

In [ ]:
tests = workdir / "tests"
tests.mkdir()

(tests / "test_stats.py").write_text('''
import pytest
from demo_pkg.stats import mean


def test_mean_of_simple_list():
    assert mean([1, 2, 3]) == 2


def test_mean_of_one_value():
    assert mean([10]) == 10


def test_mean_of_empty_list_raises():
    with pytest.raises(ValueError):
        mean([])
'''.lstrip())

(workdir / "pytest.ini").write_text("[pytest]\npythonpath = src\n")

run(sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider")

**What should you test?** Start with the functions where a silent mistake would hurt most: cleaning functions
(does it handle missing values?), anything with a formula, and anything with boundaries (is 20 °C "warm" or "mild"?).
A good habit: **when you find a bug, write a test that fails because of it, then fix the bug.**

## 8. The refactoring recipe for *your* project

Do this with your own notebook, one step at a time, running the notebook after each step:

1. **Restart & Run All.** Make sure the notebook works top to bottom before touching it.
2. **Pick one block of cells** that does one job (e.g. cleaning). Wrap it in a function in the right module (`clean.py`).
3. **Replace the cells** with an import and a call: `from myproject.clean import clean`.
4. **Run again.** The results must be identical.
5. **Add one or two tests** for that function; commit.
6. **Repeat** for loading, features, plots, analysis.
7. Create **`run_analysis.py`** that calls the same functions in order, so a single command reproduces everything.

### Common errors
| message | usual cause |
|---|---|
| `ModuleNotFoundError: No module named 'myproject'` | package not installed (`pip install -e .`) or wrong environment |
| `ImportError: cannot import name 'x'` | typo, or the function lives in a different module |
| circular import | two modules import each other; move shared code into a third module |
| `FileNotFoundError` | path built relative to the wrong place; use `Path(__file__)` |
| change not taking effect | module already imported; restart the kernel or use `%autoreload 2` |

## Recap

* Notebooks to explore, modules to build, tests to trust.
* `if __name__ == "__main__"` · packages with `__init__.py` · `src/` layout · `pip install -e .`
* Functions: one job, arguments in, `return` out, no mutation, docstring.
* `pathlib` for paths · `argparse` for options · `pytest` for confidence.

**Next:** the hands-on exercise in [`exercise/`](exercise/README.md): make 13 failing tests pass. Then do the
**project checkpoint** in the README.